## Introduction

Companion notebook to UMBC's CMSC 412/612 Neurosymbolic Text Generation.

Throughout this homework, you are welcome to change any hyperparamters to your liking just as long as you have two finetuned models in the end (one for each dataset) and evaluate them with BLEU & ROUGE. For the best comparison, it helps to have the two models as similar as possible with only conditioned data changed between the two.

## Installations
 Install the necessary libraries:

In [5]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9\.]{3,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.32.post2" if v == "2.8.0" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1,<4.0.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.55.4
!pip install --no-deps trl==0.22.2


**Important: You will probably need to restart your session after installing the libraries above.**

## Getting the Data

This is a modified dataset from the [Plan, Write, and Revise paper](https://aclanthology.org/N19-4016/). It contains the 5-sentence stories, their extracted keywords, and their corresponding titles. We will only be looking at 20 stories from this dataset.

I have written functions below to extract the relevant data.
* `load_data` will return a list of all of the data in the file.
* `get_story` will return a list of the sentences in the story.
* `get_title` will return the title of a story from a given line.
* `get_keywords` will return the keywords of a story from a given line/sentence. Lines are numbered 0-4. Each line may have multiple keywords.

In [ ]:
!wget https://raw.githubusercontent.com/lara-martin/interactive-fiction-class/refs/heads/master/homeworks/plan-and-write/keyword-story.txt

--2026-09-23 22:02:37--  https://raw.githubusercontent.com/lara-martin/interactive-fiction-class/refs/heads/master/homeworks/plan-and-write/keyword-story.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10325286 (9.8M) [text/plain]
Saving to: ‘keyword-story.txt’

keyword-story.txt   100%[===================>]   9.85M  --.-KB/s    in 0.09s   

2026-09-23 22:02:37 (104 MB/s) - ‘keyword-story.txt’ saved [10325286/10325286]



In [52]:
#fixing my car <EOT> car turn tested alternator bad <EOL> </s> i went to start my car last friday . </s> my car would n't turn over . </s> i took my alternator off to be tested . </s> the parts store said that it was bad . </s> i replaced my alternator with a new one .

from collections import defaultdict

def load_data():
  test = []
  train = []
  with open('keyword-story.txt','r') as in_file:
    reader = in_file.readlines()
    for i, line in enumerate(reader[1:]):
      if i < len(reader)/10:
        test.append(line.strip())
      else:
        train.append(line.strip())
  return train,test

def get_title(line):
  title, _ = line.split(" <EOT> ")
  return title

def combine_words(sentence):
  s = sentence.strip().replace(" .",".").replace(" n't","n't").replace(" '","'").replace(" ,",",").replace(" i "," I ").replace(" !","!")
  return s.upper()[0]+s[1:]

def get_story(line):
  _, story = line.split(" <EOL>")
  sentences = [sentence.strip() for sentence in story.split(" </s>") if sentence.strip() != ""]
  return sentences


def get_keywords(line):
  _, rest = line.split(" <EOT> ")
  keywords,_ = rest.split(" <EOL> ")
  keydict = defaultdict(list)
  index = 0
  for keyword in keywords.split():
    if keyword == "#":
      index+=1
    else:
      keydict[index].append(keyword)
  return keydict

train_stories, test_stories = load_data()

In [41]:
len(test_stories)

2966

In [53]:
# Here are some print statements to show what each of these looks like
print(get_keywords(train_stories[0])) #dictionary of keywords for each sentence; keys are sentence index, values are lists of keywords
print(get_title(train_stories[0]))
print(get_story(train_stories[0]))

defaultdict(<class 'list'>, {0: ['pregnant'], 1: ['today', 'learned', 'twins'], 2: ['husband', 'military'], 3: ['scared'], 4: ['happy', 'healthy']})
Little Boys
['Kim is pregnant for the first time in her life.', 'Today she learned she is having twins!', "Her husband is in the military and won't be home before they come.", 'She is not scared though because her mom has come to stay with her.', 'Soon the babies arrive and they are happy and healthy little boys!']


## Setup the model

This next step will take a couple minutes to download the model, although we will be using a library (Unsloth) that will load it faster.

If you are curious about the model you'll be using, you can check out the [Qwen3-0.6B documentation on HuggingFace](https://huggingface.co/Qwen/Qwen3-0.6B)

In [2]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import (
    get_chat_template,
)
import torch

MODEL_ID = "unsloth/Qwen3-0.6B"
QAT_SCHEME = "int8-int4"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_ID,
    max_seq_length = 2048,
    dtype = torch.bfloat16,
    load_in_4bit = False,
    full_finetuning = True,
    qat_scheme = QAT_SCHEME,
)
tokenizer = get_chat_template(tokenizer, chat_template = "qwen3")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.13/dist-packages/unsloth_zoo/__init__.py:547: UserWarning: Unsloth fused-forward install skipped: requires transformers >= 4.56.0.
  _install_fused_forward()


🦥 Unsloth Zoo will now patch everything to make training faster!


==((====))==  Unsloth 2026.9.11: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Not planning a device map; full finetuning does not use the quantized planner. Using `sequential`.
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.
Unsloth: Applying QAT to mitigate quantization degradation


## Setup the data

Before we get to finetuning, you need to setup the data. In `format_controlled_data` and `format_uncontrolled_data` below, change the `user` and `assistant` variables to provide the model with a different prompt.
You will need to setup two sets of data because you will be training two models: one for the controlled story generation (given a title and keywords, generate the story) and one for the uncontrolled story generation (given a title, generate the story).
The methods will take in the `keywords`, `title`, and stories (`og_story`) from `train_stories`, you just need to format it in a way such that the user is giving the task and the assitant is providing the answer (aka the story).

Follow the instructions on [the homework page](https://laramartin.net/neurosymbolic-text-gen/homeworks/plan-and-write/plan-and-write.html) for more information on what you should be generating.

The blocks below will help you format the training data for finetuning the models.

**TODO:** You just need to change the value of the `user` and `assistant` variables in both of the following methods.

In [20]:
def format_controlled_data(examples):
    conversations = []

    for story in examples:
      title  = get_title(story)
      keywords = get_keywords(story)
      og_story = get_story(story)
      # TODO: change the user and assistant variables below to pass information to the LLM as if the user is speaking to the LLM assistant
      user = ""
      assistant = ""
      conversations.append([
            {"role" : "user",      "content" : user},
            {"role" : "assistant", "content" : assistant},
        ])
    return { "conversations": conversations, }

In [ ]:
def format_uncontrolled_data(examples):
    conversations = []

    for story in examples:
      title  = get_title(story)
      og_story = get_story(story)
      # TODO: change the user and assistant variables below to pass information to the LLM as if the user is speaking to the LLM assistant
      user = ""
      assistant = ""
      conversations.append([
            {"role" : "user",      "content" : user},
            {"role" : "assistant", "content" : assistant},
        ])
    return { "conversations": conversations, }

Then just run this block to get the data ready.

In [43]:
controlled_story_dataset = tokenizer.apply_chat_template(
    list(format_controlled_data(train_stories)["conversations"]),tokenize = False,
)
uncontrolled_story_dataset = tokenizer.apply_chat_template(
    list(format_uncontrolled_data(train_stories)["conversations"]),tokenize = False,
)

## Finetune the models

Now that the pretrained model and the data are ready, use [HuggingFace's supervised finetuning trainer (SFT Trainer)](https://huggingface.co/docs/trl/sft_trainer) to finetune Qwen to generate stories. Optionally, since you are already installing Unsloth, you can use their SFT integration: https://huggingface.co/docs/trl/unsloth_integration

**Big TODO:** Implement finetuning using SFT Trainer, your story datasets, and the pretrained `model`.

In [ ]:
# TODO: finetune "model" twice, once for each of the two datasets (controlled and uncontrolled)
finetuned_model_uncontrolled = None
finetuned_model_controlled = None

**Tip:** You will probably want to save your models to your Google Drive so that you access them later.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
# Optional TODO: If you want to save your models, modify the directory above to
# where you want to save it and save the model to that directory.

# Evaluation

Now you will compare your the unguided generated stories to the original stories and compare the guided generated stories to the original stories using the following libraries:

BLEU -
[https://www.nltk.org/api/nltk.translate.bleu_score.html](https://www.nltk.org/api/nltk.translate.bleu_score.html)
- Run *modified n-gram precision* with unigrams and bigrams

ROUGE -
[https://pypi.org/project/rouge-score/](https://pypi.org/project/rouge-score/)
- Run unigrams, bigrams, and ROUGE-L




### Generate test data

First you need to generate the test data using your finetuned models.

**TODO:** Set the `user` content to match the prompt format from your training data.

In [44]:
def decodeUncontrolled(title):
  #TODO: edit the "content" of "messages" to use the title for generating the story.
  # Be sure to match your "user" format from your data setup above.
  messages = [
    {"role" : "user", "content" : "CHANGE THIS"}
  ]

  text = tokenizer.apply_chat_template(
      messages,
      tokenize = False,
      add_generation_prompt = True,
      enable_thinking = False,
  )

  from transformers import TextStreamer
  outputs = model.generate(
      **tokenizer(text, return_tensors = "pt").to("cuda"),
      max_new_tokens = 256,
      temperature = 0.7, top_p = 0.8, top_k = 20,
      streamer = TextStreamer(tokenizer, skip_prompt = True),
  )

  return tokenizer.batch_decode(outputs)


Example of what the output will look like:

In [47]:
decodeUncontrolled("A long time ago in a galaxy far, far away")

As a thought-provoking and imaginative narrative, here's a short story:

In a distant galaxy far away, time and space had not yet settled, and the stars were filled with mysteries. In one of the great civilizations, a great leader, Sira, who was known for his wisdom and curiosity, lived in a time when knowledge was not just a gift but a power. He was tasked with helping the people of the galaxy to understand their surroundings and to find the answers they had long been seeking.

One day, as Sira walked through a hidden city, he noticed a strange anomaly in the sky. The stars were not the same as they were before, and the planets were not in their natural positions. With a heart full of curiosity, Sira decided to investigate. He traveled through time, and he found a message from a long time ago. The message said:

*"In the galaxy far away, time is not just a passage, but a force that shapes the stars and the planets. To understand the world, one must listen to the echoes of ancient time

['<|im_start|>user\nA long time ago in a galaxy far, far away<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\nAs a thought-provoking and imaginative narrative, here\'s a short story:\n\nIn a distant galaxy far away, time and space had not yet settled, and the stars were filled with mysteries. In one of the great civilizations, a great leader, Sira, who was known for his wisdom and curiosity, lived in a time when knowledge was not just a gift but a power. He was tasked with helping the people of the galaxy to understand their surroundings and to find the answers they had long been seeking.\n\nOne day, as Sira walked through a hidden city, he noticed a strange anomaly in the sky. The stars were not the same as they were before, and the planets were not in their natural positions. With a heart full of curiosity, Sira decided to investigate. He traveled through time, and he found a message from a long time ago. The message said:\n\n*"In the galaxy far away, time is not just a pass

In [6]:
def decodeControlled(title, keywords):
  #TODO: edit the "content" of "messages" to use the title and keywords for generating the story
  # Be sure to match your "user" format from your data setup above.
  messages = [
    {"role" : "user", "content" : "CHANGE THIS"}
  ]"

  text = tokenizer.apply_chat_template(
      messages,
      tokenize = False,
      add_generation_prompt = True,
      enable_thinking = False,
  )

  from transformers import TextStreamer
  outputs = model.generate(
      **tokenizer(text, return_tensors = "pt").to("cuda"),
      max_new_tokens = 256,
      temperature = 0.7, top_p = 0.8, top_k = 20,
      streamer = TextStreamer(tokenizer, skip_prompt = True),
  )

  return tokenizer.batch_decode(outputs)


Run the following block to iterate through the test set of stories and call your prompts.

In [ ]:
uncontrolled_stories = []
controlled_stories = []

for story in test_stories:
  decoded_text = decodeUncontrolled(get_title(story),finetuned_model_uncontrolled)
  uncontrolled_stories.append(decoded_text)
  decoded_text = decodeControlled(get_title(story),get_keywords(story),finetuned_model_controlled)
  controlled_stories.append(decoded_text)

**TODO:** Now you will need to implement BLEU and ROGUE. Feel free to use the libraries linked to above.

In [ ]:
import statistics
def BLEU(hypothesis, target, n=1):
  # TODO: calculate BLEU by comparing the generated story (hypothesis)
  # to the original story (target) looking at n-grams for multiple n's
  # return the average score across the sentences of the stories
  return

def ROUGE(hypothesis, target, n="L"):
  # TODO: calculate ROUGE by comparing the generated story (hypothesis)
  # to the original story (target) looking at n-grams for multiple n's AND ROUGE-L
  # return the average score across the sentences of the stories
  return


Run this to get your results:

In [ ]:
avg_BLEU_1_controlled = []
avg_BLEU_2_controlled = []
avg_BLEU_1_uncontrolled = []
avg_BLEU_2_uncontrolled = []

avg_ROUGE_1_controlled = []
avg_ROUGE_2_controlled = []
avg_ROUGE_L_controlled = []
avg_ROUGE_1_uncontrolled = []
avg_ROUGE_2_uncontrolled = []
avg_ROUGE_L_uncontrolled = []


# Iterate through BLEU and ROUGE for all test stories & average the scores
for i, line in enumerate(test_stories):
  print(get_story(line))
  print(controlled_stories[i])
  avg_BLEU_1_controlled.append(BLEU(get_story(line),controlled_stories[i], 1))
  avg_BLEU_2_controlled.append(BLEU(get_story(line),controlled_stories[i], 2))
  avg_BLEU_1_uncontrolled.append(BLEU(get_story(line),uncontrolled_stories[i], 1))
  avg_BLEU_2_uncontrolled.append(BLEU(get_story(line),uncontrolled_stories[i], 2))

  avg_ROUGE_1_controlled.append(ROUGE(get_story(line),controlled_stories[i], "1"))
  avg_ROUGE_2_controlled.append(ROUGE(get_story(line),controlled_stories[i], "2"))
  avg_ROUGE_L_controlled.append(ROUGE(get_story(line),controlled_stories[i], "L"))
  avg_ROUGE_1_uncontrolled.append(ROUGE(get_story(line),uncontrolled_stories[i], "1"))
  avg_ROUGE_2_uncontrolled.append(ROUGE(get_story(line),uncontrolled_stories[i], "2"))
  avg_ROUGE_L_uncontrolled.append(ROUGE(get_story(line),uncontrolled_stories[i], "L"))


print("\t\t| BLEU-1\t| BLEU-2\t| ROUGE-1\t| ROUGE-2\t| ROUGE-L\t|")
print("Controlled\t| {}\t\t| {}\t\t| {}\t\t| {}\t\t| {}\t\t|".format(statistics.mean(avg_BLEU_1_controlled),statistics.mean(avg_BLEU_2_controlled),statistics.mean(avg_ROUGE_1_controlled),statistics.mean(avg_ROUGE_2_controlled),statistics.mean(avg_ROUGE_L_controlled)))
print("Uncontrolled\t| {}\t\t| {}\t\t| {}\t\t| {}\t\t| {}\t\t|".format(statistics.mean(avg_BLEU_1_uncontrolled),statistics.mean(avg_BLEU_2_uncontrolled),statistics.mean(avg_ROUGE_1_uncontrolled),statistics.mean(avg_ROUGE_2_uncontrolled),statistics.mean(avg_ROUGE_L_uncontrolled)))

Now go back to the [homework page](https://laramartin.net/neurosymbolic-text-gen/homeworks/plan-and-write/plan-and-write.html) and answer the questions about your results in a separate document.

Unsloth code adapted from https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/AMD-Qwen3_(0.6B)-Reasoning-Conversational-ExecuTorch.ipynb
